# Part A — Exploratory Survival Analysis

This notebook renders the figures for Part A from the parquet files
written by `part_a_survival_analysis.py`. All figures live here; the
compute script does no plotting.

**Estimand.** The Kaplan–Meier curves below estimate the *cause-specific*
survival function for prepayment. Loans terminated for any reason other
than prepayment (credit-related sale, REO, whole-loan sale, etc.) are
treated as **non-informative censoring** at their termination month. If
those competing exits are correlated with prepayment through borrower
quality or rate environment, this assumption is approximate (see
concern 10 in `01_concerns_response.md`); a Fine–Gray competing-risks
model is a candidate Part E extension.

**On Nelson–Aalen.** The cumulative-hazard plot below shows two curves
that are often confused: $\hat\Lambda(t) = -\ln\hat S_{\text{KM}}(t)$
and the Nelson–Aalen estimator $\hat\Lambda_{\text{NA}}(t) = \sum_{t_i \le t} d_i/n_i$.
They are asymptotically equal but differ slightly in finite samples
(concern 11).


In [ ]:
import sys
from pathlib import Path

# Make `import utilities` work when notebook is launched from results_a/
NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if NB_DIR.name == "results_a" else NB_DIR
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from utilities import (
    RESULTS_A,
    FICO_BUCKETS, LTV_BUCKETS, VINTAGE_BUCKETS, PURPOSE_LABELS,
    FICO_COLORS, LTV_COLORS, VINTAGE_COLORS, PURPOSE_COLORS,
    apply_plot_style,
)

apply_plot_style()


In [ ]:
# Load all the parquet outputs Part A produced
summary = pd.read_parquet(RESULTS_A / "summary.parquet")
vintage_summary = pd.read_parquet(RESULTS_A / "vintage_summary.parquet")
km_overall = pd.read_parquet(RESULTS_A / "km_overall.parquet")
hazard = pd.read_parquet(RESULTS_A / "hazard_overall.parquet")
km_by_fico = pd.read_parquet(RESULTS_A / "km_by_fico.parquet")
km_by_ltv = pd.read_parquet(RESULTS_A / "km_by_ltv.parquet")
km_by_vintage = pd.read_parquet(RESULTS_A / "km_by_vintage.parquet")
km_by_purpose = pd.read_parquet(RESULTS_A / "km_by_purpose.parquet")
medians = pd.read_parquet(RESULTS_A / "stratum_medians.parquet")

print("Loaded:")
for name, df in [
    ("summary", summary), ("vintage_summary", vintage_summary),
    ("km_overall", km_overall), ("hazard", hazard),
    ("km_by_fico", km_by_fico), ("km_by_ltv", km_by_ltv),
    ("km_by_vintage", km_by_vintage), ("km_by_purpose", km_by_purpose),
    ("medians", medians),
]:
    print(f"  {name}: {len(df):>6,} rows, columns = {list(df.columns)}")


## Dataset summary

The censoring breakdown matters for interpreting the KM curves: every
loan that hasn't prepaid by the end of its observation window is
censored, regardless of *why* (active, credit-terminated, or
other-terminated). This addresses concern A1 from the earlier critique.


In [ ]:
print(summary.to_string(index=False))
print()
print("By vintage:")
print(vintage_summary.to_string(index=False))


## A(i) — Overall Kaplan–Meier survival curve


In [ ]:
median_t = km_overall.attrs.get("median_survival_time", None)
# attrs don't survive parquet round-trips; recover from the curve directly
if median_t is None:
    cross = km_overall[km_overall["S"] <= 0.5]
    median_t = float(cross["t"].iloc[0]) if len(cross) else np.nan

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(km_overall["t"], km_overall["S"], color="#2563EB", linewidth=1.8,
        label="Prepayment")
ax.fill_between(km_overall["t"], km_overall["S_lower"], km_overall["S_upper"],
                color="#2563EB", alpha=0.15, label="95% CI")
ax.axvline(x=320, color="#DC2626", linestyle=":", alpha=0.9,
           linewidth=1.4, label="t = 320 mo")
ax.set_title("Kaplan-Meier Survival Curve: Mortgage Prepayment\n"
             "(30-Year Fixed Rate Mortgages, Freddie Mac)")
ax.set_xlabel("Loan Age (months)")
ax.set_ylabel("Survival Probability (not yet prepaid)")
ax.set_xlim(0, 360)
ax.set_ylim(0, 1.02)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, linewidth=0.8)
if np.isfinite(median_t):
    ax.axvline(x=median_t, color="gray", linestyle="--",
               alpha=0.5, linewidth=0.8)
    ax.annotate(f"Median: {median_t:.0f} mo",
                xy=(median_t, 0.5),
                xytext=(median_t + 15, 0.55),
                fontsize=10, color="gray",
                arrowprops=dict(arrowstyle="->", color="gray", alpha=0.7))
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()


In [ ]:
# Survival probability at key timepoints
key_t = [12, 24, 36, 60, 84, 120, 180, 240]
rows = []
for t in key_t:
    s_t = float(km_overall.loc[km_overall["t"] == t, "S"].iloc[0])
    rows.append({"t (months)": t, "S(t)": s_t,
                 "% prepaid": 100 * (1 - s_t)})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))


## A(ii) — Implied hazard rates

Two related quantities:

- **Discrete monthly hazard** $h(t) = 1 - S(t)/S(t-1)$, the conditional
  probability of prepaying in month $t$ given survival to month $t$.
- **Cumulative hazard** $\Lambda(t)$ — shown two ways for comparison
  (concern 11):
  - $\hat\Lambda(t) = -\ln \hat S_{\text{KM}}(t)$ — the value
    implied by the KM survival curve.
  - $\hat\Lambda_{\text{NA}}(t)$ — the Nelson–Aalen estimator
    proper.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Discrete hazard (raw + smoothed)
ax = axes[0]
ax.bar(hazard["t"], hazard["h_discrete"], width=1, color="#2563EB",
       alpha=0.4, label="Monthly hazard")
ax.plot(hazard["t"], hazard["h_smoothed"], color="#DC2626", linewidth=1.5,
        label="6-month moving avg")
ax.set_title("Discrete Monthly Hazard: Prepayment")
ax.set_xlabel("Loan Age (months)")
ax.set_ylabel("h(t) = P(prepay at t | survived to t)")
ax.set_xlim(0, 360)
ax.legend()

# Cumulative hazard — both versions
ax = axes[1]
ax.plot(hazard["t"], hazard["cum_hazard_km"], color="#2563EB", linewidth=1.6,
        label=r"$-\ln \hat S_{KM}(t)$")
ax.plot(hazard["t"], hazard["cum_hazard_na"], color="#DC2626", linewidth=1.2,
        linestyle="--", label=r"Nelson-Aalen $\hat\Lambda_{NA}(t)$")
ax.set_title("Cumulative Hazard")
ax.set_xlabel("Loan Age (months)")
ax.set_ylabel(r"$\Lambda(t)$")
ax.set_xlim(0, 360)
ax.legend(loc="lower right")

fig.suptitle("Prepayment Hazard Analysis (30-Year FRM, Freddie Mac)",
             fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

# Quantitative gap between the two cumulative-hazard curves
diff = (hazard["cum_hazard_km"] - hazard["cum_hazard_na"]).abs()
print(f"|Λ_KM − Λ_NA| over t=1..360:  max={diff.max():.4f}, "
      f"mean={diff.mean():.4f}")
print("Both estimate the same quantity; finite-sample disagreement is"
      " O(1/n).")


In [ ]:
key_t = [6, 12, 24, 36, 60, 84, 120]
rows = []
for t in key_t:
    h = float(hazard.loc[hazard["t"] == t, "h_discrete"].iloc[0])
    annualized = 100 * (1 - (1 - h) ** 12)
    rows.append({"t (months)": t, "h(t)": h,
                 "annualized %": annualized})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.6f}"))


## A(iii) — Stratified survival curves

Survival curves stratified by FICO score, original LTV, vintage, and
loan purpose.


In [ ]:
def plot_stratified(km_df, bucket_order, color_list, title, legend_prefix=""):
    """Render one stratified KM panel."""
    fig, ax = plt.subplots(figsize=(10, 6))
    for bucket, color in zip(bucket_order, color_list):
        sub = km_df[km_df["stratum"] == str(bucket)]
        if len(sub) == 0:
            continue
        n = int(sub["n_in_stratum"].iloc[0])
        ax.plot(sub["t"], sub["S"], color=color, linewidth=1.5,
                label=f"{legend_prefix}{bucket} (n={n:,})")
    ax.set_title(title)
    ax.set_xlabel("Loan Age (months)")
    ax.set_ylabel("Survival Probability")
    ax.set_xlim(0, 360)
    ax.set_ylim(0, 1.02)
    ax.legend(loc="lower left")
    fig.tight_layout()
    plt.show()


In [ ]:
plot_stratified(
    km_by_fico, FICO_BUCKETS, FICO_COLORS,
    "Prepayment Survival by FICO Score\n(30-Year FRM, Freddie Mac)",
)

# Median table
m = medians[medians["variable"] == "FICO"].sort_values("stratum",
    key=lambda s: s.map({b: i for i, b in enumerate(FICO_BUCKETS)}))
print(m[["stratum", "n", "n_events", "median_survival_time", "prepay_rate"]]
      .to_string(index=False))


In [ ]:
plot_stratified(
    km_by_ltv, LTV_BUCKETS, LTV_COLORS,
    "Prepayment Survival by Original LTV\n(30-Year FRM, Freddie Mac)",
    legend_prefix="LTV ",
)
m = medians[medians["variable"] == "LTV"].sort_values("stratum",
    key=lambda s: s.map({b: i for i, b in enumerate(LTV_BUCKETS)}))
print(m[["stratum", "n", "n_events", "median_survival_time", "prepay_rate"]]
      .to_string(index=False))


In [ ]:
plot_stratified(
    km_by_vintage, VINTAGE_BUCKETS, VINTAGE_COLORS,
    "Prepayment Survival by Origination Vintage\n(30-Year FRM, Freddie Mac)",
)
m = medians[medians["variable"] == "Vintage"].sort_values("stratum",
    key=lambda s: s.map({b: i for i, b in enumerate(VINTAGE_BUCKETS)}))
print(m[["stratum", "n", "n_events", "median_survival_time", "prepay_rate"]]
      .to_string(index=False))


In [ ]:
purpose_codes = list(PURPOSE_LABELS.keys())
purpose_color_list = [PURPOSE_COLORS[c] for c in purpose_codes]

# Render with friendly labels rather than 1-letter codes
fig, ax = plt.subplots(figsize=(10, 6))
for code in purpose_codes:
    sub = km_by_purpose[km_by_purpose["stratum"] == code]
    if len(sub) == 0:
        continue
    n = int(sub["n_in_stratum"].iloc[0])
    ax.plot(sub["t"], sub["S"], color=PURPOSE_COLORS[code], linewidth=1.5,
            label=f"{PURPOSE_LABELS[code]} (n={n:,})")
ax.set_title("Prepayment Survival by Loan Purpose\n(30-Year FRM, Freddie Mac)")
ax.set_xlabel("Loan Age (months)")
ax.set_ylabel("Survival Probability")
ax.set_xlim(0, 360)
ax.set_ylim(0, 1.02)
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

m = medians[medians["variable"] == "LoanPurpose"]
m = m.assign(label=m["stratum"].map(PURPOSE_LABELS))
print(m[["stratum", "label", "n", "n_events",
         "median_survival_time", "prepay_rate"]]
      .to_string(index=False))


## Caveats and limitations

- **Cause-specific framing.** The KM estimator above treats credit-related
  terminations and other non-prepayment exits as non-informative
  censoring. Concern 10.
- **Cumulative-hazard relabel.** What used to be called "Nelson-Aalen"
  in earlier versions of this analysis was actually
  $-\ln \hat S_{\text{KM}}$. The plot above shows both estimators
  side-by-side. Concern 11.
- **Right-censoring is varied.** Some loans are censored because they
  remain active; others because of credit-related termination or
  whole-loan sale. The summary table above breaks this down, addressing
  the misleading "right-censored" count in earlier output (concern A1).
